Installing libraries

In [ ]:
!pip install torch torchvision albumentations timm bayesian-optimization

Cloning model's repository

In [ ]:
!rm -rf /content/MNv4-fpn
!git clone https://github.com/xwd2019/MNv4-fpn.git

Imports

In [ ]:
import torch
from torch.utils.data import DataLoader, Subset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
import os
from google.colab import drive
import sys
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import cv2
from skimage.restoration import rolling_ball, denoise_wavelet
from skimage.morphology import white_tophat, black_tophat, disk
from skimage.filters import gaussian
from skimage import exposure
from bayes_opt import BayesianOptimization
import json

Fixing model's imports

In [ ]:
!cd /content/MNv4-fpn && sed -i 's/from \./from /g' *.py
sys.path.insert(0, '/content/MNv4-fpn')
from seg import Seg

Mounting and data pairing

In [ ]:
drive.mount('/content/drive')
os.makedirs('/content/drive/MyDrive/microplastics/results', exist_ok=True)

Calculating mean and std

In [ ]:
def calculate_dataset_stats(image_files, images_path):
    means = []
    stds = []

    for img_file in image_files:
        img = Image.open(os.path.join(images_path, img_file)).convert('RGB')
        img = np.array(img) / 255.0

        means.append(img.mean(axis=(0, 1)))
        stds.append(img.std(axis=(0, 1)))

    dataset_mean = np.mean(means, axis=0)
    dataset_std = np.mean(stds, axis=0)

    return dataset_mean.tolist(), dataset_std.tolist()

Dataset configuration

In [ ]:
DATASET = 'ftmc'

if DATASET == 'ftmc':
    images_path = '/content/drive/MyDrive/microplastics/Mikroplastikai_dataset/images'
    masks_path = '/content/drive/MyDrive/microplastics/Mikroplastikai_dataset/masks'
    image_files = sorted(os.listdir(images_path))
    mask_files = sorted(os.listdir(masks_path))
    paired_image_files = [img for img in image_files if img in set(mask_files)]
     # Precomputed using calculate_dataset_stats()
    MEAN = [0.7059140755261906, 0.7096701363145624, 0.7028037838470678]
    STD = [0.04292269066694413, 0.04979341462785254, 0.04669112573302777]

elif DATASET == 'mdpi':
    base_path = '/content/drive/MyDrive/microplastics/papildomas_dataset/data'
    train_images_path = f'{base_path}/0_train'
    train_masks_path = f'{base_path}/0_train_bi'
    val_images_path = f'{base_path}/1_valid'
    val_masks_path = f'{base_path}/1_valid_bi'
    test_images_path = f'{base_path}/2_test'
    test_masks_path = f'{base_path}/2_test_bi'
     # Precomputed using calculate_dataset_stats()
    MEAN = [0.3089483468151623, 0.2881815555622238, 0.2820614605766218]
    STD = [0.3269065497598735, 0.3352709155239046, 0.3346825753796247]

print(f"Dataset: {DATASET}")
print(f"Mean: {MEAN}, STD: {STD}")

Dataset split

In [ ]:
DATASET = 'ftmc'

torch.manual_seed(200)
np.random.seed(200)

if DATASET == 'ftmc':
    total_samples = len(paired_image_files)
    train_size = int(0.70 * total_samples)
    val_size = int(0.15 * total_samples)
    all_indices = np.random.permutation(total_samples)
    train_image_files = [paired_image_files[i] for i in all_indices[:train_size]]
    val_image_files = [paired_image_files[i] for i in all_indices[train_size:train_size + val_size]]
    test_image_files = [paired_image_files[i] for i in all_indices[train_size + val_size:]]
    train_images_path = val_images_path = test_images_path = images_path
    train_masks_path = val_masks_path = test_masks_path = masks_path

elif DATASET == 'mdpi':
    train_image_files = sorted(os.listdir(train_images_path))
    val_image_files = sorted(os.listdir(val_images_path))
    test_image_files = sorted(os.listdir(test_images_path))

print(f"Train: {len(train_image_files)}, Val: {len(val_image_files)}, Test: {len(test_image_files)}")

Hyperparameters and configuration

In [ ]:
# Hyperparameters based on the third study (Yao et al., 2025)
BATCH_SIZE = 8
NUM_CLASSES = 2
NUM_EPOCHS = 80
LEARNING_RATE_MIN = 0
LEARNING_RATE_MAX = 0.0001
WEIGHT_DECAY = 0.0001 # Adopted from the first study (Han et al., 2023)
LOSS_ALPHA = 0.4 # For combined loss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Microplastics dataset

In [ ]:
class MicroplasticsDataset(Dataset):
    def __init__(self, images_path, masks_path, image_files, transform=None, preprocessing_fn=None):
        self.images_path = images_path
        self.masks_path = masks_path
        self.image_files = image_files
        self.transform = transform
        self.preprocessing_fn = preprocessing_fn

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_file = self.image_files[idx]

        img = Image.open(os.path.join(self.images_path, img_file)).convert('RGB')
        mask = Image.open(os.path.join(self.masks_path, img_file)).convert('L')

        img = np.array(img)
        mask = np.array(mask)

        if self.preprocessing_fn is not None:
            img = self.preprocessing_fn(img)

        mask = (mask > 127).astype(np.uint8)

        if self.transform:
            transformed = self.transform(image=img, mask=mask)
            img = transformed['image']
            mask = transformed['mask']

            if not isinstance(mask, torch.Tensor):
                mask = torch.from_numpy(mask)
            mask = mask.long()
        else:
            img = torch.from_numpy(img).permute(2, 0, 1).float()
            mask = torch.from_numpy(mask).long()

        return img, mask

Transforms

In [ ]:
train_transform = A.Compose([
    A.Resize(height=480, width=640),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Resize(height=480, width=640),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(height=480, width=640),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2()
])

Loss and metrics

In [ ]:
class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.alpha = alpha
        self.ce_loss = nn.CrossEntropyLoss()

    def dice_loss(self, pred, target):
        pred = torch.softmax(pred, dim=1)[:, 1, :, :]
        target = target.float()

        batch_size = pred.shape[0]
        dice_sum = 0

        for i in range(batch_size):
            intersection = (pred[i] * target[i]).sum()
            union = pred[i].sum() + target[i].sum()
            dice = (2. * intersection + 1e-7) / (union + 1e-7)
            dice_sum += dice

        return 1 - dice_sum / batch_size

    def forward(self, pred, target):
        ce = self.ce_loss(pred, target)
        dice = self.dice_loss(pred, target)

        return self.alpha * ce + (1 - self.alpha) * dice

def calculate_iou(pred, target, smooth=1e-7):

    pred = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    union = pred.sum() + target.sum() - intersection

    iou = (intersection + smooth) / (union + smooth)
    return iou.item()

def calculate_metrics(pred, target, smooth=1e-7):
    pred = pred.view(-1)
    target = target.view(-1)

    tp = (pred * target).sum()
    fp = (pred * (1 - target)).sum()
    fn = ((1 - pred) * target).sum()

    precision = (tp + smooth) / (tp + fp + smooth)
    recall = (tp + smooth) / (tp + fn + smooth)
    f1 = 2 * (precision * recall) / (precision + recall + smooth)

    return precision.item(), recall.item(), f1.item()

Preprocessing functions

In [ ]:
def apply_clahe(img, clip_limit=0.015, tile_grid_size=8):
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(
        clipLimit=clip_limit * 255,
        tileGridSize=(int(tile_grid_size), int(tile_grid_size))
    )
    l = clahe.apply(l)
    lab = cv2.merge([l, a, b])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def apply_he(img):
    img_yuv = cv2.cvtColor(img, cv2.COLOR_RGB2YUV)
    img_yuv[:,:,0] = cv2.equalizeHist(img_yuv[:,:,0])
    return cv2.cvtColor(img_yuv, cv2.COLOR_YUV2RGB)

def apply_gaussian(img, sigma=1.5):
    ksize = int(sigma * 6) | 1
    return cv2.GaussianBlur(img, (ksize, ksize), sigma)

def apply_wavelet(img, sigma=0.02):
    img_float = img / 255.0
    denoised = denoise_wavelet(img_float, method='BayesShrink', mode='soft',
                               channel_axis=-1, rescale_sigma=False, sigma=sigma)
    return (np.clip(denoised, 0, 1) * 255).astype(np.uint8)

def apply_grayscale(img):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

def apply_rolling_ball(img, radius=20):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    kernel_size = int(radius) * 2 + 1
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    background = cv2.morphologyEx(gray, cv2.MORPH_OPEN, kernel)
    corrected = cv2.subtract(gray, background)
    return cv2.cvtColor(corrected, cv2.COLOR_GRAY2RGB)

def apply_white_tophat(img, kernel_size=15):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (int(kernel_size)*2+1, int(kernel_size)*2+1))
    result = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel)
    result = cv2.normalize(result, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.cvtColor(result, cv2.COLOR_GRAY2RGB)

def apply_black_tophat(img, kernel_size=15):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE,
                                       (int(kernel_size)*2+1, int(kernel_size)*2+1))
    result = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    result = cv2.normalize(result, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.cvtColor(result, cv2.COLOR_GRAY2RGB)

def apply_color_jitter(img, brightness=0.2, contrast=0.2, saturation=0.2):
    transform = A.Compose([
        A.ColorJitter(brightness=brightness, contrast=contrast,
                      saturation=saturation, hue=0.1, p=1.0)
    ])
    return transform(image=img)['image']

PREPROCESSING_METHODS = {
    'baseline': None,
    'clahe': apply_clahe,
    'he': apply_he,
    'gaussian': apply_gaussian,
    'wavelet': apply_wavelet,
    'grayscale': apply_grayscale,
    'rolling_ball': apply_rolling_ball,
    'white_tophat': apply_white_tophat,
    'black_tophat': apply_black_tophat,
    'color_jitter': apply_color_jitter,
}

print("Pre-processing functions ready")

Optimization helper functions

In [ ]:
NUM_EPOCHS_OPT = 25

def create_loaders(preprocessing_fn):
    train_ds = MicroplasticsDataset(train_images_path, train_masks_path, train_image_files, transform=train_transform, preprocessing_fn=preprocessing_fn)
    val_ds = MicroplasticsDataset(val_images_path, val_masks_path, val_image_files, transform=val_test_transform, preprocessing_fn=preprocessing_fn)
    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return train_ld, val_ld

def run_experiment(preprocessing_fn, num_epochs=NUM_EPOCHS_OPT):
    torch.manual_seed(200)
    model = Seg(num_classes=NUM_CLASSES).to(device)
    criterion = CombinedLoss(alpha=LOSS_ALPHA)
    optimizer = optim.RMSprop(model.parameters(), lr=LEARNING_RATE_MAX, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=LEARNING_RATE_MIN)

    train_ld, val_ld = create_loaders(preprocessing_fn)

    best_f1 = 0.0
    for epoch in range(1, num_epochs + 1):
        train_one_epoch(model, train_ld, criterion, optimizer, device, epoch, num_epochs)
        _, _, _, _, val_f1 = validate(model, val_ld, criterion, device, epoch, num_epochs)
        scheduler.step()
        if val_f1 > best_f1:
            best_f1 = val_f1

    return best_f1

Train and validation loops

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device, epoch, num_epochs):

    model.train()

    total_loss = 0
    total_iou = 0
    num_batches = len(loader)

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{num_epochs} [TRAIN]")

    for batch_idx, (images, masks) in enumerate(pbar):
        images = images.to(device)
        masks = masks.to(device)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        preds = torch.argmax(outputs, dim=1)
        iou = calculate_iou(preds, masks)

        total_loss += loss.item()
        total_iou += iou

        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'iou': f'{iou:.4f}'
        })

    avg_loss = total_loss / num_batches
    avg_iou = total_iou / num_batches

    return avg_loss, avg_iou


def validate(model, loader, criterion, device, epoch, num_epochs):

    model.eval()

    total_loss = 0
    total_iou = 0
    total_precision = 0
    total_recall = 0
    total_f1 = 0
    num_batches = len(loader)

    pbar = tqdm(loader, desc=f"Epoch {epoch}/{num_epochs} [VAL]  ")

    with torch.no_grad():
        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss = criterion(outputs, masks)

            preds = torch.argmax(outputs, dim=1)
            iou = calculate_iou(preds, masks)
            precision, recall, f1 = calculate_metrics(preds, masks)

            total_loss += loss.item()
            total_iou += iou
            total_precision += precision
            total_recall += recall
            total_f1 += f1

            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'iou': f'{iou:.4f}',
                'f1': f'{f1:.4f}'
            })

    avg_loss = total_loss / num_batches
    avg_iou = total_iou / num_batches
    avg_precision = total_precision / num_batches
    avg_recall = total_recall / num_batches
    avg_f1 = total_f1 / num_batches

    return avg_loss, avg_iou, avg_precision, avg_recall, avg_f1


Bayesian optimization

In [ ]:
# HE and grayscale have no parameters
best_params = {}

def save_params(params, filename='/content/drive/MyDrive/microplastics/results/best_params.json'):
    with open(filename, 'w') as f:
        json.dump(params, f, indent=2)
    print(f"Parameters saved: {list(params.keys())}")

def clahe_objective(clip_limit, tile_grid_size):
    return run_experiment(lambda img: apply_clahe(img, clip_limit=clip_limit, tile_grid_size=int(tile_grid_size)))

optimizer_clahe = BayesianOptimization(f=clahe_objective, pbounds={'clip_limit': (0.001, 0.05), 'tile_grid_size': (4, 16)}, random_state=200)
optimizer_clahe.maximize(init_points=3, n_iter=7)
best_params['clahe'] = optimizer_clahe.max['params']
save_params(best_params)
print(f"CLAHE: {best_params['clahe']}")

def gaussian_objective(sigma):
    return run_experiment(lambda img: apply_gaussian(img, sigma=sigma))

optimizer_gaussian = BayesianOptimization(f=gaussian_objective, pbounds={'sigma': (0.5, 5.0)}, random_state=200)
optimizer_gaussian.maximize(init_points=3, n_iter=7)
best_params['gaussian'] = optimizer_gaussian.max['params']
save_params(best_params)
print(f"Gaussian: {best_params['gaussian']}")

def wavelet_objective(sigma):
    return run_experiment(lambda img: apply_wavelet(img, sigma=sigma))

optimizer_wavelet = BayesianOptimization(f=wavelet_objective, pbounds={'sigma': (0.005, 0.1)}, random_state=200)
optimizer_wavelet.maximize(init_points=2, n_iter=5)
best_params['wavelet'] = optimizer_wavelet.max['params']
save_params(best_params)
print(f"Wavelet: {best_params['wavelet']}")

def rolling_ball_objective(radius):
    return run_experiment(lambda img: apply_rolling_ball(img, radius=radius))

optimizer_rb = BayesianOptimization( f=rolling_ball_objective,  pbounds={'radius': (5, 50)},  random_state=200)
optimizer_rb.maximize(init_points=2, n_iter=5)
best_params['rolling_ball'] = optimizer_rb.max['params']
save_params(best_params)
print(f"Rolling ball: {best_params['rolling_ball']}")

def white_tophat_objective(kernel_size):
    return run_experiment(lambda img: apply_white_tophat(img, kernel_size=int(kernel_size)))

optimizer_wth = BayesianOptimization(f=white_tophat_objective, pbounds={'kernel_size': (5, 50)}, random_state=200)
optimizer_wth.maximize(init_points=2, n_iter=5)
best_params['white_tophat'] = optimizer_wth.max['params']
save_params(best_params)
print(f"White top-hat: {best_params['white_tophat']}")
def black_tophat_objective(kernel_size):
    return run_experiment(lambda img: apply_black_tophat(img, kernel_size=int(kernel_size)))

optimizer_bth = BayesianOptimization(f=black_tophat_objective, pbounds={'kernel_size': (5, 50)}, random_state=200)
optimizer_bth.maximize(init_points=2, n_iter=5)
best_params['black_tophat'] = optimizer_bth.max['params']
save_params(best_params)
print(f"Black top-hat: {best_params['black_tophat']}")

def color_jitter_objective(brightness, contrast, saturation):
    return run_experiment(lambda img: apply_color_jitter(img, brightness=brightness, contrast=contrast, saturation=saturation))

optimizer_cj = BayesianOptimization(f=color_jitter_objective, pbounds={'brightness': (0.1, 0.5), 'contrast': (0.1, 0.5), 'saturation': (0.1, 0.5)}, random_state=200)
optimizer_cj.maximize(init_points=3, n_iter=7)
best_params['color_jitter'] = optimizer_cj.max['params']
save_params(best_params)
print(f"Color jitter: {best_params['color_jitter']}")

with open('/content/drive/MyDrive/microplastics/results/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

print("\nBest parametres saved.")

Experiment configuration

In [ ]:
with open('/content/drive/MyDrive/microplastics/results/best_params.json', 'r') as f:
    best_params = json.load(f)

experiments = {
    'baseline': None,
    'clahe': lambda img: apply_clahe(img, **best_params['clahe']),
    'he': apply_he,
    'gaussian': lambda img: apply_gaussian(img, **best_params['gaussian']),
    'wavelet': lambda img: apply_wavelet(img, **best_params['wavelet']),
    'grayscale': apply_grayscale,
    'rolling_ball': lambda img: apply_rolling_ball(img, **best_params['rolling_ball']),
    'white_tophat': lambda img: apply_white_tophat(img, int(best_params['white_tophat']['kernel_size'])),
    'black_tophat': lambda img: apply_black_tophat(img, int(best_params['black_tophat']['kernel_size'])),
    'color_jitter': lambda img: apply_color_jitter(img, **best_params['color_jitter']),
}

all_results = {}

print("Experiments ready.")
print(f"Methods: {list(experiments.keys())}")

Preprocess and save function

In [ ]:
def preprocess_and_save(method_name, preprocessing_fn, image_files, images_path, split_name=''):
    folder = f'{method_name}_{split_name}' if split_name else method_name
    save_dir = f'/content/drive/MyDrive/microplastics/preprocessed_{DATASET}/{folder}'
    os.makedirs(save_dir, exist_ok=True)

    if len(os.listdir(save_dir)) == len(image_files):
        print(f"{method_name} already preprocessed")
        return save_dir

    for img_file in tqdm(image_files, desc=f"Preprocessing {method_name}"):
        img = np.array(Image.open(os.path.join(images_path, img_file)).convert('RGB'))
        processed = preprocessing_fn(img)
        Image.fromarray(processed).save(os.path.join(save_dir, img_file))

    print(f"{method_name} saved.")
    return save_dir

Preprocessed paths

In [ ]:
preprocessed_paths = {}

for exp_name, prep_fn in experiments.items():
    if DATASET == 'ftmc':
        if prep_fn is None:
            preprocessed_paths[exp_name] = images_path
        else:
            preprocessed_paths[exp_name] = f'/content/drive/MyDrive/microplastics/preprocessed_ftmc/{exp_name}'
    elif DATASET == 'mdpi':
        if prep_fn is None:
            preprocessed_paths[exp_name] = {
                'train': train_images_path,
                'val': val_images_path,
                'test': test_images_path
            }
        else:
            preprocessed_paths[exp_name] = {
                'train': f'/content/drive/MyDrive/microplastics/preprocessed_mdpi/{exp_name}_train',
                'val': f'/content/drive/MyDrive/microplastics/preprocessed_mdpi/{exp_name}_val',
                'test': f'/content/drive/MyDrive/microplastics/preprocessed_mdpi/{exp_name}_test'
            }

print("Preprocessed paths ready.")
print(preprocessed_paths)

Experiments' training loops

In [ ]:
for exp_name, prep_fn in experiments.items():
    print(f"Experiment: {exp_name}")

    preprocessed_images_path = preprocessed_paths[exp_name]

    torch.manual_seed(200)
    model = Seg(num_classes=NUM_CLASSES).to(device)
    criterion = CombinedLoss(alpha=LOSS_ALPHA)
    optimizer = optim.RMSprop(model.parameters(), lr=LEARNING_RATE_MAX, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=LEARNING_RATE_MIN)

    if DATASET == 'ftmc':
        train_ds = MicroplasticsDataset(preprocessed_paths[exp_name], train_masks_path, train_image_files, transform=train_transform, preprocessing_fn=None)
        val_ds = MicroplasticsDataset(preprocessed_paths[exp_name], val_masks_path, val_image_files, transform=val_test_transform, preprocessing_fn=None)
        test_ds = MicroplasticsDataset(preprocessed_paths[exp_name], test_masks_path, test_image_files, transform=val_test_transform, preprocessing_fn=None)
    elif DATASET == 'mdpi':
        train_ds = MicroplasticsDataset(preprocessed_paths[exp_name]['train'], train_masks_path, train_image_files, transform=train_transform, preprocessing_fn=None)
        val_ds = MicroplasticsDataset(preprocessed_paths[exp_name]['val'], val_masks_path, val_image_files, transform=val_test_transform, preprocessing_fn=None)
        test_ds = MicroplasticsDataset(preprocessed_paths[exp_name]['test'], test_masks_path, test_image_files, transform=val_test_transform, preprocessing_fn=None)

    train_ld = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_ld = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    test_ld = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

    save_path = f'/content/drive/MyDrive/microplastics/results/{DATASET}_{exp_name}_best_model.pth'

    best_val_f1 = 0.0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_iou': []}

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, _ = train_one_epoch(model, train_ld, criterion, optimizer, device, epoch, NUM_EPOCHS)
        val_loss, val_iou, _, _, val_f1 = validate(model, val_ld, criterion, device, epoch, NUM_EPOCHS)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_iou'].append(val_iou)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save({'model_state_dict': model.state_dict(), 'val_f1': val_f1, 'val_iou': val_iou}, save_path)
            print(f"New best model saved! F1: {val_f1:.4f}")

    checkpoint = torch.load(save_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    test_f1, test_iou, test_precision, test_recall = 0, 0, 0, 0
    with torch.no_grad():
        for images, masks in tqdm(test_ld, desc="Testing"):
            images, masks = images.to(device), masks.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            iou = calculate_iou(preds, masks)
            precision, recall, f1 = calculate_metrics(preds, masks)
            test_f1 += f1
            test_iou += iou
            test_precision += precision
            test_recall += recall

    n = len(test_ld)
    all_results[exp_name] = {
        'test_f1': test_f1 / n,
        'test_iou': test_iou / n,
        'test_precision': test_precision / n,
        'test_recall': test_recall / n,
        'best_val_f1': best_val_f1
    }

    print(f"Test F1: {test_f1/n:.4f}, IoU: {test_iou/n:.4f}")

    with open(f'/content/drive/MyDrive/microplastics/results/{DATASET}_all_results.json', 'w') as f:
        json.dump(all_results, f, indent=2)

    with open(f'/content/drive/MyDrive/microplastics/results/{DATASET}_{exp_name}_history.json', 'w') as f:
        json.dump(history, f, indent=2)

print("Results saved")

FTMC training loop visualization

In [ ]:
DATASET = 'ftmc'
methods = {
    'baseline': 'Bazinis atvejis',
    'gaussian': 'Gauso filtras',
    'white_tophat': '"White Top-hat"',
    'rolling_ball': '"Rolling-ball"'
}

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, (method, label) in enumerate(methods.items()):
    path = f'/content/drive/MyDrive/microplastics/results/{DATASET}_{method}_history.json'

    with open(path) as f:
        history = json.load(f)

    ax1 = axes[idx]


    ax1.plot(history['val_f1'], label='Validavimo F1', color='blue')
    ax1.plot(history['val_iou'], label='Validavimo IoU ', color='red', alpha=0.5)
    ax1.plot(history['val_loss'], label='Validavimo nuostolis', color='purple', alpha=0.5)
    ax1.plot(history['train_loss'], label='Apmokymo nuostolis', color='green', alpha=0.5)


    ax1.set_title(label, fontsize=11)
    ax1.set_xlabel('Epocha')
    ax1.set_ylim(0, 1)
    ax1.grid(True)

    lines1, labels1 = ax1.get_legend_handles_labels()
    ax1.legend(lines1 , labels1, fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/microplastics/results/ftmc_training_curves_selected.png', dpi=150)
plt.show()

MDPI training loop visualization

In [ ]:
DATASET = 'mdpi'
methods = {
    'baseline': 'Bazinis atvejis',
    'gaussian': 'Gauso filtras',
    'black_tophat': '"Black Top-hat"',
    'color_jitter': 'Spalvų virpinimas'
}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for idx, (method, label) in enumerate(methods.items()):
    path = f'/content/drive/MyDrive/microplastics/results/{DATASET}_{method}_history.json'

    with open(path) as f:
        history = json.load(f)

    ax1 = axes[idx]


    ax1.plot(history['val_f1'], label='Validavimo F1', color='blue')
    ax1.plot(history['val_iou'], label='Validavimo IoU ', color='red', alpha=0.5)
    ax1.plot(history['val_loss'], label='Validavimo nuostolis', color='purple', alpha=0.5)
    ax1.plot(history['train_loss'], label='Apmokymo nuostolis', color='green', alpha=0.5)


    ax1.set_title(label, fontsize=11)
    ax1.set_xlabel('Epocha')
    ax1.set_ylim(0, 1)
    ax1.grid(True)

    lines1, labels1 = ax1.get_legend_handles_labels()
    ax1.legend(lines1 , labels1, fontsize=7, loc='upper right')

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/microplastics/results/mdpi_training_curves_selected.png', dpi=150)
plt.show()

Prediction visualization function

In [ ]:
def show_prediction(dataset, method, img_idx):
    if dataset == 'ftmc':
        test_ds_orig = MicroplasticsDataset(
            preprocessed_paths['baseline'],
            test_masks_path,
            test_image_files,
            transform=val_test_transform,
            preprocessing_fn=None
        )
        test_ds = MicroplasticsDataset(
            preprocessed_paths[method],
            test_masks_path,
            test_image_files,
            transform=val_test_transform,
            preprocessing_fn=None
        )
    elif dataset == 'mdpi':
        test_ds_orig = MicroplasticsDataset(
            preprocessed_paths['baseline']['test'],
            test_masks_path,
            test_image_files,
            transform=val_test_transform,
            preprocessing_fn=None
        )
        test_ds = MicroplasticsDataset(
            preprocessed_paths[method]['test'],
            test_masks_path,
            test_image_files,
            transform=val_test_transform,
            preprocessing_fn=None
        )

    checkpoint = torch.load(f'/content/drive/MyDrive/microplastics/results/{dataset}_{method}_best_model.pth')
    m = Seg(num_classes=NUM_CLASSES).to(device)
    m.load_state_dict(checkpoint['model_state_dict'])
    m.eval()

    img_orig, mask = test_ds_orig[img_idx]
    img, _ = test_ds[img_idx]
    img_input = img.unsqueeze(0).to(device)

    with torch.no_grad():
        pred = torch.argmax(m(img_input), dim=1)[0].cpu().numpy()

    img_vis = img_orig.permute(1, 2, 0).numpy()
    img_vis = img_vis * np.array(STD) + np.array(MEAN)
    img_vis = np.clip(img_vis, 0, 1)

    img_prep = img.permute(1, 2, 0).numpy()
    img_prep = img_prep * np.array(STD) + np.array(MEAN)
    img_prep = np.clip(img_prep, 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    axes[0].imshow(img_vis)
    axes[0].set_title('Originalas')
    axes[0].axis('off')

    axes[1].imshow(img_prep)
    axes[1].set_title(f'Po apdorojimo')
    axes[1].axis('off')

    axes[2].imshow(pred, cmap='gray')
    axes[2].set_title(f'Spėjama kaukė')
    axes[2].axis('off')

    axes[3].imshow(mask.numpy(), cmap='gray')
    axes[3].set_title('Tikra kaukė')
    axes[3].axis('off')

    plt.suptitle(f'{dataset.upper()} — {method}')
    plt.tight_layout()
    plt.show()


show_prediction('ftmc', 'baseline', 1)